# 09 — Backend Smoke Test (17 September)

Local FastAPI backend over the **frozen** pipeline. Executes really:
`/health` → `/config` → `/frames` → `/replay/load` → `/replay/run` →
`/results/{id}` → `/metrics/{id}` → schema checks. Real nuScenes Mini frame,
no mocked LiDAR.

In [ ]:
# 1. Drive mount (Colab only; skipped locally)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('drive mounted')
except ImportError:
    print('local run: drive mount skipped')

In [ ]:
# 2. Dependencies
import importlib, subprocess, sys
for pkg, mod in [('fastapi','fastapi'),('uvicorn','uvicorn'),('httpx','httpx'),('nuscenes-devkit','nuscenes')]:
    try:
        importlib.import_module(mod); print(pkg, 'ok')
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg]); print(pkg, 'installed')

In [ ]:
# 3. Project on sys.path (adjust PROJECT_ROOT for Drive layouts)
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
for cand in [PROJECT_ROOT, PROJECT_ROOT/'Paradox-Protocol']:
    if (cand/'backend'/'app.py').exists():
        PROJECT_ROOT = cand; break
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# 4-11. Full smoke test through the real backend (in-process TestClient)
from fastapi.testclient import TestClient
from backend.app import app

client = TestClient(app)
FRAME = '5991fad3280c4f84b331536c32001a04'
checks = {}
def check(name, cond):
    checks[name] = 'PASS' if cond else 'FAIL'
    print(f'{name}: {checks[name]}')

r = client.get('/health'); check('health', r.status_code==200 and r.json()['status']=='ok')
r = client.get('/config'); check('config', r.status_code==200 and r.json()['max_mapping_distance_m']==100.0)
r = client.get('/frames'); check('frames', r.status_code==200 and r.json()['count']==7)
r = client.post('/replay/load', json={'frame_id': FRAME})
check('load', r.status_code==200 and r.json()['point_count']>0)
r = client.post('/replay/run', json={'frame_id': FRAME})
check('run', r.status_code==200 and r.json()['result']['map_cell_count']>0)
res = r.json()['result']
r = client.get(f'/results/{FRAME}')
check('results', r.status_code==200 and r.json()['result']['frame_id']==FRAME)
r = client.get(f'/metrics/{FRAME}')
check('metrics', r.status_code==200 and r.json()['latency_ms'] is not None)
r = client.get('/demo/status')
check('demo_status', r.status_code==200 and r.json()['replay_available'])

# 12. Response-schema contract
cell = res['map_cells'][0]
check('contract', all(k in res for k in ('frame_id','timestamp','map_cells','importance','resolution','semantic','timing'))
      and cell['resolution'] in (0.05,0.10,0.20,0.50)
      and cell['semantic_source'] in ('annotation','fallback','lidarseg','lidarseg_annotation','object_annotation','unknown','model'))
print('cells:', res['map_cell_count'], '| latency_ms:', res['timing']['total_latency_ms'], '| fps:', res['timing']['fps'])
print('semantic:', res['semantic']['source_counts'])
assert all(v=='PASS' for v in checks.values()), checks

## 13. Cleanup / standalone server

In-process test needs no cleanup. For the standalone SIH server instead:
`uvicorn backend.app:app --host 127.0.0.1 --port 8000`, then `GET
http://127.0.0.1:8000/health`. Stop with Ctrl+C.